# 17_SHAP_Analysis_Lab_Candidate_Selection
## Materia Arche V3.2
### SHAP explainability on locked ET model + 3-candidate lab panel selection

Production model locked in NB15 (ExtraTrees, tau-b 0.2714, CV 0.2889).
This notebook:
1. SHAP feature importance on the locked model
2. SHAP dependence plots for top features
3. Select 3 candidates for the Phase 2 lab panel
4. Package candidate shortlist for outreach attachment

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"Libraries loaded (shap {shap.__version__})")

Libraries loaded (shap 0.49.1)


## 1. Reproduce locked model

In [2]:
# Load data and reproduce locked model from NB15
df = pd.read_csv("perovskite_stability_clean.csv")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Locked production model (NB15 best params)
et_prod = ExtraTreesRegressor(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42
)
et_prod.fit(X_train, y_train)
pred = et_prod.predict(X_test)
tau, _ = kendalltau(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print(f"Locked model reproduced: tau-b = {tau:.4f}, MAE = {mae:.4f}")

Locked model reproduced: tau-b = 0.2714, MAE = 1.2521


## 2. SHAP feature importance

In [3]:
# SHAP TreeExplainer on the locked ExtraTrees model
explainer = shap.TreeExplainer(et_prod)

# Compute SHAP values on test set
shap_values = explainer.shap_values(X_test)

# Mean absolute SHAP importance
shap_importance = pd.DataFrame({
    'Feature': ml_features,
    'Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_Abs_SHAP', ascending=False)

print("=" * 65)
print("SHAP FEATURE IMPORTANCE (locked ExtraTrees model)")
print("=" * 65)
print(shap_importance.to_string(index=False))

# SHAP summary plot
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=ml_features, show=False)
plt.tight_layout()
plt.savefig("shap_summary_locked_ET.png", dpi=150, bbox_inches='tight')
plt.close()
print("\nshap_summary_locked_ET.png saved")

SHAP FEATURE IMPORTANCE (locked ExtraTrees model)
                            Feature  Mean_Abs_SHAP
                     JV_default_Jsc       0.150602
                Perovskite_band_gap       0.132188
                     JV_default_Voc       0.096943
                 Cell_area_measured       0.092457
                      JV_default_FF       0.088508
first_Prvskt_thermal_annealing_time       0.086705
               Perovskite_thickness       0.085801
 first_Prvskt_annealing_temperature       0.081570
                                 Cs       0.048296
                                 Br       0.041436
                                 MA       0.040707
                                 FA       0.039298
                                  I       0.028855
                                 Pb       0.016640
                                 Sn       0.015700
                                 Cl       0.000492

shap_summary_locked_ET.png saved


## 3. SHAP dependence plots

In [4]:
# SHAP dependence plots for top 4 features
top4 = shap_importance.head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, feat in enumerate(top4):
    ax = axes[i // 2][i % 2]
    shap.dependence_plot(
        feat, shap_values, X_test, feature_names=ml_features,
        ax=ax, show=False
    )
    ax.set_title(f"SHAP dependence: {feat}")

plt.tight_layout()
plt.savefig("shap_dependence_top4.png", dpi=150, bbox_inches='tight')
plt.close()
print("shap_dependence_top4.png saved")
print(f"\nTop 4 features by SHAP importance: {top4}")

shap_dependence_top4.png saved

Top 4 features by SHAP importance: ['JV_default_Jsc', 'Perovskite_band_gap', 'JV_default_Voc', 'Cell_area_measured']


## 4. Select 3 candidates for lab panel

In [5]:
# Select 3 candidates for the Phase 2 lab panel
# Strategy: literature baseline + classical best + ML-predicted best
# All from the actual dataset — these are real compositions

# Train production model on ALL data for final rankings
et_full = ExtraTreesRegressor(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42
)
et_full.fit(X, y)
df['predicted_log_T80'] = et_full.predict(X)
df['predicted_T80_hours'] = np.expm1(df['predicted_log_T80'])

# Candidate 1: Literature baseline — median stability composition
median_idx = (df['Stability_PCE_T80'] - df['Stability_PCE_T80'].median()).abs().idxmin()
candidate_baseline = df.loc[median_idx]

# Candidate 2: Best actual T80 in dataset (known good, for calibration)
best_actual_idx = df['Stability_PCE_T80'].idxmax()
candidate_actual_best = df.loc[best_actual_idx]

# Candidate 3: Highest ML-predicted T80 that differs from actual best
# (i.e., a composition the model thinks is good but hasn't been top-ranked before)
df_sorted = df.sort_values('predicted_log_T80', ascending=False)
# Skip if it's the same composition as actual best
for idx, row in df_sorted.iterrows():
    if idx != best_actual_idx:
        candidate_ml_best = row
        ml_best_idx = idx
        break

cols_show = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
             'MA', 'FA', 'Cs', 'Stability_PCE_T80', 'predicted_T80_hours']

panel = pd.DataFrame([
    candidate_baseline[cols_show],
    candidate_actual_best[cols_show],
    candidate_ml_best[cols_show]
], index=['Baseline (median)', 'Best actual T80', 'ML-predicted best'])

panel['Role'] = ['Control/calibration', 'Known best (validation)', 'ML discovery candidate']

print("=" * 75)
print("3-CANDIDATE LAB PANEL FOR PHASE 2")
print("=" * 75)
print(panel.to_string())
print("")
print("Panel design:")
print("  Candidate 1 (Baseline): Median-stability composition — control")
print("  Candidate 2 (Best actual): Highest measured T80 — calibration")
print("  Candidate 3 (ML pick): Highest predicted T80 (different from #2)")
print("")
print("If ML pick outperforms baseline: ML adds value")
print("If ML pick matches best actual: ML correctly identifies winners")
print("If ML pick underperforms: ML needs more data/features")

3-CANDIDATE LAB PANEL FOR PHASE 2
                   Perovskite_band_gap   Pb   Sn     I    Br   Cl    MA    FA   Cs  Stability_PCE_T80  predicted_T80_hours                     Role
Baseline (median)                 1.60  1.0  0.0  3.00  0.00  0.0  1.00  0.00  0.0            150.384           147.915593      Control/calibration
Best actual T80                   1.59  1.0  0.0  2.55  0.45  0.0  0.15  0.85  0.0           8400.000          3480.702292  Known best (validation)
ML-predicted best                 1.60  1.0  0.0  2.55  0.45  0.0  0.15  0.85  0.0           5400.000          5373.965843   ML discovery candidate

Panel design:
  Candidate 1 (Baseline): Median-stability composition — control
  Candidate 2 (Best actual): Highest measured T80 — calibration
  Candidate 3 (ML pick): Highest predicted T80 (different from #2)

If ML pick outperforms baseline: ML adds value
If ML pick matches best actual: ML correctly identifies winners
If ML pick underperforms: ML needs more data/featur

## 5. Candidate shortlist for outreach

In [6]:
# Package candidate shortlist for outreach attachment
shortlist = panel.copy()
shortlist['SHAP_top_driver'] = ''

# For each candidate, find the top SHAP driver
for idx_name, idx_val in [('Baseline (median)', median_idx),
                           ('Best actual T80', best_actual_idx),
                           ('ML-predicted best', ml_best_idx)]:
    sv = explainer.shap_values(X.iloc[[idx_val]])
    top_feat_idx = np.abs(sv[0]).argmax()
    shortlist.loc[idx_name, 'SHAP_top_driver'] = (
        f"{ml_features[top_feat_idx]} (SHAP={sv[0][top_feat_idx]:+.3f})"
    )

print("=" * 75)
print("CANDIDATE SHORTLIST FOR OUTREACH ATTACHMENT")
print("=" * 75)
print(shortlist[['Perovskite_band_gap', 'Pb', 'I', 'MA', 'FA', 'Cs',
                 'Stability_PCE_T80', 'predicted_T80_hours', 'Role',
                 'SHAP_top_driver']].to_string())

shortlist.to_csv("Lab_Candidate_Shortlist.csv", index=False)
print("\nLab_Candidate_Shortlist.csv exported")

# Also save SHAP importance
shap_importance.to_csv("SHAP_Feature_Importance.csv", index=False)
print("SHAP_Feature_Importance.csv exported")

CANDIDATE SHORTLIST FOR OUTREACH ATTACHMENT
                   Perovskite_band_gap   Pb     I    MA    FA   Cs  Stability_PCE_T80  predicted_T80_hours                     Role                     SHAP_top_driver
Baseline (median)                 1.60  1.0  3.00  1.00  0.00  0.0            150.384           147.915593      Control/calibration        JV_default_Jsc (SHAP=+0.128)
Best actual T80                   1.59  1.0  2.55  0.15  0.85  0.0           8400.000          3480.702292  Known best (validation)                    FA (SHAP=+0.536)
ML-predicted best                 1.60  1.0  2.55  0.15  0.85  0.0           5400.000          5373.965843   ML discovery candidate  Perovskite_thickness (SHAP=+0.587)



Lab_Candidate_Shortlist.csv exported
SHAP_Feature_Importance.csv exported


## 6. Summary

In [7]:
print("=" * 65)
print("SUMMARY — NOTEBOOK 17")
print("=" * 65)
print("")
print("SHAP ANALYSIS:")
print(f"  Top feature: {shap_importance.iloc[0]['Feature']}")
print(f"    Mean |SHAP|: {shap_importance.iloc[0]['Mean_Abs_SHAP']:.4f}")
print(f"  Second: {shap_importance.iloc[1]['Feature']}")
print(f"    Mean |SHAP|: {shap_importance.iloc[1]['Mean_Abs_SHAP']:.4f}")
print(f"  Third: {shap_importance.iloc[2]['Feature']}")
print(f"    Mean |SHAP|: {shap_importance.iloc[2]['Mean_Abs_SHAP']:.4f}")
print("")
print("LAB PANEL (3 candidates x 5 devices each = 15 devices):")
for idx_name in panel.index:
    row = panel.loc[idx_name]
    print(f"  {idx_name}:")
    print(f"    Composition: Pb={row['Pb']}, I={row['I']}, MA={row['MA']}, FA={row['FA']}, Cs={row['Cs']}")
    print(f"    Actual T80: {row['Stability_PCE_T80']:.0f}h, Predicted: {row['predicted_T80_hours']:.0f}h")
    print(f"    Role: {row['Role']}")
print("")
print("DELIVERABLES:")
print("  - shap_summary_locked_ET.png (feature importance beeswarm)")
print("  - shap_dependence_top4.png (top 4 feature dependence)")
print("  - Lab_Candidate_Shortlist.csv (attach to outreach email)")
print("  - SHAP_Feature_Importance.csv")
print("")
print("Phase 2 status: READY TO SEND")
print("  Outreach templates: NB12")
print("  Lab candidate shortlist: this notebook")
print("  Evidence package: NB16")
print("")
print("Notebooks: 17 (01-17)")

SUMMARY — NOTEBOOK 17

SHAP ANALYSIS:
  Top feature: JV_default_Jsc
    Mean |SHAP|: 0.1506
  Second: Perovskite_band_gap
    Mean |SHAP|: 0.1322
  Third: JV_default_Voc
    Mean |SHAP|: 0.0969

LAB PANEL (3 candidates x 5 devices each = 15 devices):
  Baseline (median):
    Composition: Pb=1.0, I=3.0, MA=1.0, FA=0.0, Cs=0.0
    Actual T80: 150h, Predicted: 148h
    Role: Control/calibration
  Best actual T80:
    Composition: Pb=1.0, I=2.55, MA=0.15, FA=0.85, Cs=0.0
    Actual T80: 8400h, Predicted: 3481h
    Role: Known best (validation)
  ML-predicted best:
    Composition: Pb=1.0, I=2.55, MA=0.15, FA=0.85, Cs=0.0
    Actual T80: 5400h, Predicted: 5374h
    Role: ML discovery candidate

DELIVERABLES:
  - shap_summary_locked_ET.png (feature importance beeswarm)
  - shap_dependence_top4.png (top 4 feature dependence)
  - Lab_Candidate_Shortlist.csv (attach to outreach email)
  - SHAP_Feature_Importance.csv

Phase 2 status: READY TO SEND
  Outreach templates: NB12
  Lab candidate short